# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Priyansh-rath18/flyrank-internship-/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

Feature vector is built from `data/raw/content_refresh_anonymized.csv` (30,000 rows, 32 clients). Steps: log1p transforms on heavy-tailed traffic totals (impressions/clicks/sessions/ai_sessions), binary flags for zero-signal rows (`has_clicks`, `has_ai_sessions`, `measurable_opportunity`), missingness flags recorded BEFORE filling (search_volume, competition, cpc, word_count, char_count, avg_position), then numeric fills to 0 and categorical fills to "unknown" matching the documented prep step. Categoricals (`content_type`, `main_intent`, `competition_level`, tiers) are one-hot encoded. `content_id`/`client_id` are kept only for grouping, never as features.

In [5]:
import pandas as pd
import numpy as np

RAW_URL = "https://raw.githubusercontent.com/Priyansh-rath18/flyrank-internship-/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(RAW_URL)

id_cols = ["content_id", "client_id"]
label_source_cols = ["trend_direction", "trend_pct"]

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

log_source_cols = ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]
for col in log_source_cols: df[f"log_{col}"] = np.log1p(df[col])

df["has_clicks"] = (df["clicks_90d"] > 0).astype(int)
df["has_ai_sessions"] = (df["ai_sessions_90d"] > 0).astype(int)
df["measurable_opportunity"] = ((df["impressions_90d"] >= 100) & (df["sessions_90d"] > 0)).astype(int)

missing_flag_cols = ["search_volume", "competition", "cpc", "word_count", "char_count"]
missing_before_fill = {}
for col in missing_flag_cols: missing_before_fill[col] = df[col].isna().mean(); df[f"has_{col}"] = df[col].notna().astype(int)

df["has_position_data"] = (df["avg_position"] > 0).astype(int)
missing_before_fill["avg_position"] = (df["avg_position"] == 0).mean()

numeric_fill_cols = ["search_volume", "competition", "cpc", "word_count", "char_count", "avg_position"]
df[numeric_fill_cols] = df[numeric_fill_cols].fillna(0)

categorical_fill_cols = ["competition_level", "content_type", "main_intent", "provider_used", "model_used", "age_tier", "freshness_tier", "word_count_tier", "char_count_tier", "impression_tier", "position_tier"]
df[categorical_fill_cols] = df[categorical_fill_cols].fillna("unknown")

feature_categoricals = ["content_type", "main_intent", "competition_level", "age_tier", "freshness_tier", "impression_tier", "position_tier"]
df_features = pd.get_dummies(df, columns=feature_categoricals, prefix=feature_categoricals)

print("raw shape:", df.shape, "| feature-encoded shape:", df_features.shape)
df_features.head(3)

raw shape: (30000, 58) | feature-encoded shape: (30000, 80)


,content_id,client_id,search_volume,competition,cpc,word_count,char_count,provider_used,model_used,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,engaged_sessions_90d,ai_sessions_90d,scroll_events_90d,days_with_impressions,days_with_sessions,impressions_last_30d,clicks_last_30d,sessions_last_30d,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,content_age_days,age_tier_order,days_since_last_update,word_count_tier,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,trend_direction,trend_pct,is_declining_label,log_impressions_90d,log_clicks_90d,log_sessions_90d,log_ai_sessions_90d,has_clicks,has_ai_sessions,measurable_opportunity,has_search_volume,has_competition,has_cpc,has_word_count,has_char_count,has_position_data,content_type_comparison article,content_type_feedly article,content_type_keyword article,main_intent_commercial,main_intent_informational,main_intent_navigational,main_intent_transactional,main_intent_unknown,competition_level_HIGH,competition_level_LOW,competition_level_MEDIUM,competition_level_unknown,age_tier_181-365,age_tier_31-90,age_tier_365+,age_tier_91-180,freshness_tier_0-30,freshness_tier_181+,freshness_tier_31-90,freshness_tier_91-180,impression_tier_excellent,impression_tier_good,impression_tier_low,impression_tier_moderate,position_tier_deep,position_tier_page_1,position_tier_page_3_5,position_tier_striking,position_tier_top_3
0,content_304f48230142,client_f369cb89fc,10.0,0.67,2.05,3221.0,20457.0,unknown,gemini-2.5-flash,3803,29,22,17,16,1,0,1,88,13,578,2,2,987,13,9,187,5,20,2000-3500,15000-25000,0.76,10.6,5.88,4.55,0.0,down,-41.4,1,8.243808,3.401197,2.890372,0.0,1,0,1,1,1,1,1,1,1,False,False,True,False,False,False,True,False,True,False,False,False,True,False,False,False,True,False,False,False,False,True,False,False,False,False,False,True,False
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,0.05,2481.0,15562.0,unknown,gemini-3-flash-preview,15320,7,10,9,9,0,0,1,88,9,2501,2,3,5915,1,2,445,6,25,2000-3500,15000-25000,0.05,20.3,0.00,10.00,0.0,down,-57.7,1,9.636980,2.079442,2.302585,0.0,1,0,1,1,1,1,1,1,1,False,False,True,False,True,False,False,False,False,True,False,False,False,False,True,False,True,False,False,False,False,True,False,False,False,False,True,False,False
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,0.00,3515.0,23643.0,unknown,gemini-2.5-flash,12581,11,14,11,11,0,0,4,88,11,2382,1,1,6089,3,3,141,4,20,3500+,15000-25000,0.09,36.5,0.00,28.57,0.0,down,-60.9,1,9.440023,2.484907,2.484907,0.0,1,0,1,1,1,1,1,1,1,False,False,True,False,True,False,False,False,False,True,False,False,False,False,False,True,True,False,False,False,False,True,False,False,False,False,True,False,False


## 2. Feature notes (meaning, missing, categorical, available-when?)

| Feature group | Meaning | Missing handling | Available before prediction? |
|---|---|---|---|
| log_impressions_90d, log_clicks_90d, log_sessions_90d, log_ai_sessions_90d | log1p of trailing-90-day GSC/GA4 totals | source totals have no nulls in this slice | Yes - trailing 90d window closes before the label's last-30d window |
| has_clicks, has_ai_sessions, measurable_opportunity | zero-signal flags from the 90d totals | never missing | Yes |
| search_volume, competition, cpc | keyword-context metrics | blank when no keyword data (follows content_type); has_search_volume/has_competition/has_cpc recorded BEFORE fillna(0) | Yes - keyword metadata set at publish time |
| word_count, char_count | article length | blank when not measured; has_word_count/has_char_count recorded BEFORE fillna(0) | Yes |
| avg_position | mean GSC position over 90d | 0 means "no data", not rank zero; has_position_data flag added before the 0-fill | Yes |
| content_type, main_intent, competition_level | categorical context, one-hot encoded | filled "unknown" | Yes |
| age_tier, freshness_tier, impression_tier, position_tier | bucketed/tiered versions of raw signals, one-hot encoded | filled "unknown" | Yes - describe current 90d state, not a future outcome |
| provider_used, model_used | which LLM generated the article | filled "unknown" but NOT one-hot encoded into df_features | Excluded - dictionary marks these "not a model feature" (see Section 4) |
| content_id, client_id | pseudonymous identifiers | never missing | Excluded from features - grouping/joins only (see Section 4) |

In [6]:
missing_report = (pd.Series(missing_before_fill) * 100).sort_values(ascending=False)
print("Missingness BEFORE fill, in percent:")
print(missing_report.round(2))

excluded_always = id_cols + label_source_cols + ["is_declining_label", "provider_used", "model_used"]
feature_cols_final = [c for c in df_features.columns if c not in excluded_always]
label_leak_present = any(c in feature_cols_final for c in label_source_cols)

print("\nLabel-source columns leaked into feature set:", label_leak_present)
print("provider_used/model_used leaked into feature set:", any(c in feature_cols_final for c in ["provider_used", "model_used"]))
print("Total candidate feature columns:", len(feature_cols_final))

Missingness BEFORE fill, in percent:
char_count       25.66
word_count       25.66
competition       8.23
search_volume     8.23
cpc               8.23
avg_position      4.02
dtype: float64

Label-source columns leaked into feature set: False
provider_used/model_used leaked into feature set: False
Total candidate feature columns: 73


## 3. The leakage hunt

Three checks against the taxonomy. (1) Label-derived: `is_declining_label` is computed from `trend_direction`, which is computed from `trend_pct`, which is computed from `impressions_last_30d` vs `impressions_prev_30d`. All four are siblings of the label and are excluded as features. The cell below trains a grouped-CV model WITH `trend_pct` mixed into an otherwise-honest feature set, then WITHOUT it - the AUC collapse from near-1.0 to a normal-looking score is the confession the skill describes. (2) Future/overlapping windows: every candidate feature is a trailing-90-day aggregate ending at export time, the same moment the label is drawn from - so the 30-day sub-windows that feed the label (`impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d`) are dropped too, since they ARE the label's inputs, not independent signal. (3) Product/decision flags: `provider_used`/`model_used` describe how the article was produced, not a ranking-system decision, and the dictionary already marks them "not a model feature" - they are excluded on that basis, not used as a baseline. The base rate is printed next to both AUCs so the lift is judged against a naive baseline, not in isolation.

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

window_sibling_cols = ["impressions_last_30d", "clicks_last_30d", "sessions_last_30d", "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]
honest_numeric = ["log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d", "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct", "search_volume", "competition", "cpc", "word_count", "char_count", "content_age_days", "days_since_last_update", "has_clicks", "has_ai_sessions", "measurable_opportunity"]
y = df["is_declining_label"]
groups = df["client_id"]

def grouped_auc(feature_cols): X = df[feature_cols].fillna(0); gkf = GroupKFold(n_splits=5); fold_aucs = [roc_auc_score(y.iloc[te], make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X.iloc[tr], y.iloc[tr]).predict_proba(X.iloc[te])[:, 1]) for tr, te in gkf.split(X, y, groups)]; return np.mean(fold_aucs)

auc_honest = grouped_auc(honest_numeric)
auc_with_trend_pct = grouped_auc(honest_numeric + ["trend_pct"])
auc_with_siblings = grouped_auc(honest_numeric + window_sibling_cols)
base_rate = y.mean()

print(f"Base rate (share declining): {base_rate:.3f}")
print(f"Grouped-CV AUC, honest features only: {auc_honest:.3f}")
print(f"Grouped-CV AUC, WITH trend_pct added (label-derived leak): {auc_with_trend_pct:.3f}")
print(f"Grouped-CV AUC, WITH last30/prev30 siblings added (window leak): {auc_with_siblings:.3f}")

Base rate (share declining): 0.542
Grouped-CV AUC, honest features only: 0.676
Grouped-CV AUC, WITH trend_pct added (label-derived leak): 0.999
Grouped-CV AUC, WITH last30/prev30 siblings added (window leak): 0.905


## 4. What I excluded and why

| Field(s) | Why excluded |
|---|---|
| `content_id`, `client_id` | Pseudonymous IDs - grouping/joins and the GroupKFold split only, never a feature (dictionary: "grouping/joins only"). |
| `trend_direction`, `trend_pct` | Direct label source - `is_declining_label` is computed from these. Using them is the textbook label-derived leak. |
| `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d` | The exact inputs to `trend_pct`/`trend_direction` - siblings of the label, not independent evidence, even though they're technically "available before" prediction time. |
| `is_declining_label` | The label itself. |
| `provider_used`, `model_used` | Dictionary marks both "Not a model feature." They describe content production, not the ranking/decline signal, and were left out of the encoded feature set entirely. |
| `age_tier_order` | Perfectly redundant with `age_tier` (just its numeric rank) - dropped to avoid a duplicate, zero-information column rather than a leakage risk. |
| `word_count_tier`, `char_count_tier` | Redundant bucketed copies of `word_count`/`char_count`, which are already features - kept the continuous version, dropped the tier to avoid double-counting the same signal. |

In [8]:
excluded_fields = {"content_id": "pseudonymous ID - grouping/joins only", "client_id": "pseudonymous ID - grouping/joins only (GroupKFold key)", "trend_direction": "direct label source", "trend_pct": "direct label source", "impressions_last_30d": "sibling input to trend_pct/trend_direction", "clicks_last_30d": "sibling input to trend_pct/trend_direction", "sessions_last_30d": "sibling input to trend_pct/trend_direction", "impressions_prev_30d": "sibling input to trend_pct/trend_direction", "clicks_prev_30d": "sibling input to trend_pct/trend_direction", "sessions_prev_30d": "sibling input to trend_pct/trend_direction", "is_declining_label": "the label itself", "provider_used": "dictionary marks it not a model feature", "model_used": "dictionary marks it not a model feature", "age_tier_order": "redundant with age_tier", "word_count_tier": "redundant with word_count", "char_count_tier": "redundant with char_count"}

final_model_features = [c for c in df_features.columns if c not in excluded_fields]
leaked = [c for c in excluded_fields if c in final_model_features]

print("Fields explicitly excluded:", len(excluded_fields))
print("Any excluded field that leaked into the final feature list:", leaked)
print("Final modeling feature count:", len(final_model_features))
print(sorted(final_model_features)[:15], "...")

Fields explicitly excluded: 16
Any excluded field that leaked into the final feature list: []
Final modeling feature count: 64
['age_tier_181-365', 'age_tier_31-90', 'age_tier_365+', 'age_tier_91-180', 'ai_sessions_90d', 'ai_traffic_pct', 'avg_position', 'char_count', 'clicks_90d', 'competition', 'competition_level_HIGH', 'competition_level_LOW', 'competition_level_MEDIUM', 'competition_level_unknown', 'content_age_days'] ...


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.